# J3-B — Extraire des entités sans casser les labels

**NLP Avancé (BERT, GPT, Hugging Face) · M2 IA · Jour 3 · 100 min · Chrys NIONGOLO**

Travail en binôme : alterner la personne qui conduit et celle qui explique. Enregistrer une copie dans Drive, puis conserver le notebook exécuté et le JSON final.

## À la fin, vous saurez

- Annoter des entités et distinguer mot et sous-token.
- Aligner BIO avec le tokenizer.
- Évaluer des entités complètes plutôt qu’une accuracy dominée par O.

## Parcours de l’après-midi

| Étape | Minutes |
| Annotation, BIO et données | 20 |
| Alignement et test | 25 |
| Entraînement guidé | 25 |
| Évaluation, erreurs et restitution | 30 |

Les durées sont des budgets d’activité incluant lecture, discussion, essais et débrief ; ce ne sont pas des durées de calcul promises. Les corpus intégrés sont originaux, fictifs et minuscules. Ils servent à comprendre un protocole, pas à certifier une qualité métier.

**Règle d’expérience :** écrire une prédiction avant de lancer une variante ; changer un seul facteur ; choisir sur la validation ; ouvrir le test après ce choix.


## 0. Préparer Colab

Importer ce fichier dans https://colab.research.google.com/ via Fichier → Importer.
Choisir Exécution → Modifier le type d’exécution → Python 3 → GPU T4 si disponible.
La disponibilité du T4 dépend des quotas Colab. Aucun compte Hugging Face n’est requis
pour les modèles publics utilisés ici. Une connexion internet est nécessaire pour les
bibliothèques et les poids ; les données pédagogiques sont déjà dans le notebook.

Exécuter la cellule suivante **avant les autres imports**. Si elle demande un redémarrage,
choisir Exécution → Redémarrer la session puis relancer depuis le début. Ne pas réinstaller
un autre PyTorch dans un runtime CUDA fonctionnel. Les versions sont volontairement fixées.
Aucun entraînement n’est lancé par cette cellule. Aucun envoi au Hub n’est automatique.

**Runpod L4 facultatif :** ce notebook fonctionne dans Jupyter avec Python 3.10+
et PyTorch CUDA préinstallé. Les chemins sont relatifs ; aucun import Google Colab
n’est requis. Arrêter le Pod après sauvegarde des résultats pour interrompre sa facturation.


In [ ]:
PINNED = {'numpy': '2.1.3', 'pandas': '2.2.3', 'matplotlib': '3.10.6', 'scikit-learn': '1.7.2', 'transformers': '4.56.2', 'tokenizers': '0.22.0', 'huggingface-hub': '0.35.3', 'accelerate': '1.10.1', 'datasets': '4.1.1', 'seqeval': '1.2.2'}
import sys, subprocess, importlib.metadata as metadata
assert sys.version_info >= (3, 10), "Utiliser Python 3.10 ou ultérieur."
module_names = {"scikit-learn": "sklearn", "sentence-transformers": "sentence_transformers",
                "huggingface-hub": "huggingface_hub", "rouge-score": "rouge_score"}
to_install = []
loaded_mismatch = []
for package, version in PINNED.items():
    module = sys.modules.get(module_names.get(package, package))
    if module is not None and getattr(module, "__version__", version) != version:
        loaded_mismatch.append(package)
    try:
        current = metadata.version(package)
    except metadata.PackageNotFoundError:
        current = None
    if current != version:
        to_install.append(f"{package}=={version}")
if to_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *to_install])
if loaded_mismatch:
    raise RuntimeError("Redémarrer la session Colab, puis relancer depuis le début : "
                       + ", ".join(loaded_mismatch))
print("Versions de référence installées. Continuer avec la cellule suivante.")


In [ ]:
import os, json, time, random, platform, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
versions = {p: metadata.version(p) for p in PINNED}
versions["python"] = platform.python_version()
results = {"seed": SEED, "versions": versions, "validation": "exécuté par le binôme"}
OUTPUT = Path("resultats")
OUTPUT.mkdir(exist_ok=True)
display(pd.Series(versions, name="version effective"))


In [ ]:
import torch
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
versions["torch"] = torch.__version__
results["device"] = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"
print("Périphérique :", results["device"])
if DEVICE == "cuda":
    print("VRAM totale (Gio) :", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
else:
    print("Pas de GPU : les sections indiquées restent utilisables ; ne pas présenter un modèle non entraîné comme adapté.")


## 1. L’annotation est une décision

PER désigne une personne, ORG une organisation, LOC un lieu et REF une référence métier.
Tous les noms et tickets sont fictifs. B commence une entité, I la poursuit, O est hors entité.
Une référence n’est pas une personne ; « Groupe Saphir » forme une organisation à deux mots.

Annoter sur papier « Le ticket RC-102 de Amadou Simon vient de Rouen. » avant de lire la sortie.
Ici, les deux formulations du même scénario restent ensemble ; même les entités changent entre groupes.
Le corpus de 24 phrases est volontairement très petit : l’objectif principal est l’alignement correct.


In [ ]:
MARKED = [('[PER:Alice Morel] attend le colis [REF:AB-120] à [LOC:Lyon].', 'Le suivi [REF:AB-120] concerne [PER:Alice Morel], à [LOC:Lyon].'), ('[ORG:Atelier Azur] a reçu la facture [REF:FA-231] à [LOC:Nantes].', 'La facture [REF:FA-231] est destinée à [ORG:Atelier Azur] dans [LOC:Nantes].'), ('Pour [PER:Samir Diallo], expédier [REF:ZX-305] vers [LOC:Bordeaux].', '[PER:Samir Diallo] demande le suivi [REF:ZX-305] pour [LOC:Bordeaux].'), ('À [LOC:Lille], [ORG:Studio Cobalt] conteste le dossier [REF:RT-410].', 'Le dossier [REF:RT-410] de [ORG:Studio Cobalt] vient de [LOC:Lille].'), ('Le compte de [PER:Inès Leroy] est utilisé chez [ORG:Maison Opale].', '[ORG:Maison Opale] signale une demande pour [PER:Inès Leroy].'), ('Retour [REF:RE-522] : [PER:Noé Martin] dépose le paquet à [LOC:Rennes].', 'À [LOC:Rennes], le retour de [PER:Noé Martin] porte [REF:RE-522].'), ('[ORG:Équipe Silex] cherche [PER:Fatou Traoré] pour le ticket [REF:TK-618].', 'Le ticket [REF:TK-618] mentionne [PER:Fatou Traoré] et [ORG:Équipe Silex].'), ('[PER:Hugo Petit] répond depuis [LOC:Toulouse] au sujet de [REF:CC-719].', 'Depuis [LOC:Toulouse], réponse de [PER:Hugo Petit] pour [REF:CC-719].'), ('Merci de transmettre à [PER:Leïla Bernard] le reçu [REF:FC-820] de [ORG:Bureau Corail].', 'Le reçu [REF:FC-820] de [ORG:Bureau Corail] est pour [PER:Leïla Bernard].'), ('Destination [LOC:Grenoble] pour [ORG:Collectif Hêtre], référence [REF:EN-921].', '[ORG:Collectif Hêtre] à [LOC:Grenoble] attend [REF:EN-921].'), ('La réclamation [REF:RC-102] est portée par [PER:Amadou Simon] de [LOC:Rouen].', '[PER:Amadou Simon], basé à [LOC:Rouen], relance [REF:RC-102].'), ('Joindre [PER:Chloé Laurent] chez [ORG:Groupe Saphir] au sujet du suivi [REF:SU-203].', '[ORG:Groupe Saphir] confie le suivi [REF:SU-203] à [PER:Chloé Laurent].')]
import re
from datasets import Dataset
entity_types = ["PER", "ORG", "LOC", "REF"]
tag_names = ["O"] + [prefix + "-" + kind for kind in entity_types for prefix in ["B", "I"]]
tag2id = {tag: i for i, tag in enumerate(tag_names)}
def parse_marked(marked):
    text, spans, last = "", [], 0
    for match in re.finditer(r"\[(PER|ORG|LOC|REF):([^\]]+)\]", marked):
        text += marked[last:match.start()]
        start = len(text)
        text += match.group(2)
        spans.append((start, len(text), match.group(1)))
        last = match.end()
    text += marked[last:]
    pieces = list(re.finditer(r"\w+(?:[-']\w+)*|[^\w\s]", text))
    tags = []
    for word in pieces:
        entity = next((s for s in spans if s[0] <= word.start() and word.end() <= s[1]), None)
        tags.append("O" if entity is None else ("B-" if word.start() == entity[0] else "I-") + entity[2])
    return {"text": text, "tokens": [w.group() for w in pieces], "ner_tags": [tag2id[t] for t in tags]}
ner_rows = []
for group, pair in enumerate(MARKED):
    for sentence in pair:
        row = parse_marked(sentence)
        row.update(group=group, split="train" if group < 8 else "validation" if group < 10 else "test")
        ner_rows.append(row)
ner_frame = pd.DataFrame(ner_rows)
display(pd.crosstab(ner_frame.group, ner_frame.split))
display(pd.DataFrame({"mot": ner_rows[0]["tokens"], "label": [tag_names[x] for x in ner_rows[0]["ner_tags"]]}))
for a in ["train", "validation", "test"]:
    for c in ["train", "validation", "test"]:
        if a != c:
            assert set(ner_frame[ner_frame.split == a].group).isdisjoint(ner_frame[ner_frame.split == c].group)


## 2. Aligner sur les sous-tokens

Un mot peut être découpé en plusieurs sous-tokens. Nous entraînons seulement le **premier sous-token**
de chaque mot ; les suivants et les tokens spéciaux reçoivent -100, valeur ignorée par la loss.
À l’évaluation, nous ne gardons que ces premiers sous-tokens pour reconstruire la séquence de labels.
Cette convention est simple, mais ce n’est pas la seule possible.

**Checkpoint.** Inspecter la référence AB-120 et les noms accentués. Le nombre de labels supervisés
doit correspondre au nombre de mots d’origine, tant qu’aucun texte n’est tronqué.


In [ ]:
MODEL_ID = 'distilbert/distilbert-base-multilingual-cased'
from transformers import (AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification,
                          TrainingArguments, Trainer, set_seed)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
assert tokenizer.is_fast
def align(batch):
    encoded = tokenizer(batch["tokens"], truncation=True, is_split_into_words=True, max_length=128)
    all_labels = []
    for i, word_labels in enumerate(batch["ner_tags"]):
        previous, aligned = None, []
        for word_id in encoded.word_ids(batch_index=i):
            aligned.append(-100 if word_id is None or word_id == previous else word_labels[word_id])
            previous = word_id
        all_labels.append(aligned)
    encoded["labels"] = all_labels
    return encoded
ner_ds = {}
for split in ["train", "validation", "test"]:
    table = ner_frame[ner_frame.split == split]
    original = Dataset.from_list(table.to_dict(orient="records"))
    ner_ds[split] = original.map(align, batched=True, remove_columns=original.column_names)
example = ner_rows[0]
encoded = tokenizer(example["tokens"], is_split_into_words=True)
aligned = align({"tokens": [example["tokens"]], "ner_tags": [example["ner_tags"]]})["labels"][0]
display(pd.DataFrame({"sous_token": tokenizer.convert_ids_to_tokens(encoded["input_ids"]),
                      "word_id": encoded.word_ids(),
                      "label_loss": ["IGNORÉ" if v == -100 else tag_names[v] for v in aligned]}))
assert sum(x != -100 for x in aligned) == len(example["tokens"])
assert all(len(tokenizer(r["tokens"], is_split_into_words=True)["input_ids"]) <= 128 for r in ner_rows)


### Repères formateur — éléments de correction

Attribuer O aux tokens spéciaux créerait une supervision artificielle ; -100 les exclut de la cross-entropy. Répéter B sur tous les sous-tokens produirait plusieurs débuts d’entités. Une autre stratégie peut propager B puis I, mais doit être cohérente jusqu’à l’évaluation. L’assertion de longueur empêche ici de masquer silencieusement une troncature d’entité.


## 3. Entraîner puis mesurer des entités entières

Seqeval vérifie les séquences BIO. Le mode strict IOB2 exige la bonne frontière et le bon type.
Prédire O partout peut donner une accuracy de tokens apparemment raisonnable, tout en n’extrayant rien.
Comparer explicitement ce témoin au modèle. Le GPU accélère l’entraînement ; sur CPU, le calcul des
métriques sur les annotations et l’inspection de l’alignement restent réalisables.


In [ ]:
from seqeval.metrics import f1_score as entity_f1, precision_score as entity_precision, recall_score as entity_recall
from seqeval.scheme import IOB2
def ner_metrics(prediction):
    ids = np.argmax(prediction.predictions, axis=-1)
    true = [[tag_names[int(y)] for y in row if y != -100] for row in prediction.label_ids]
    pred = [[tag_names[int(p)] for p, y in zip(row_p, row_y) if y != -100]
            for row_p, row_y in zip(ids, prediction.label_ids)]
    return {"entity_f1": entity_f1(true, pred, mode="strict", scheme=IOB2, zero_division=0),
            "entity_precision": entity_precision(true, pred, mode="strict", scheme=IOB2, zero_division=0),
            "entity_recall": entity_recall(true, pred, mode="strict", scheme=IOB2, zero_division=0)}
example_true = [["B-PER", "I-PER", "O", "B-LOC"]]
example_partial = [["B-PER", "O", "O", "B-LOC"]]
print("F1 entité parfaite :", entity_f1(example_true, example_true, mode="strict", scheme=IOB2))
print("F1 avec frontière tronquée :", entity_f1(example_true, example_partial, mode="strict", scheme=IOB2))
true_test = [[tag_names[i] for i in r["ner_tags"]] for r in ner_rows if r["split"] == "test"]
zeros = [["O"] * len(row) for row in true_test]
print("Baseline tout-O F1 entité :", entity_f1(true_test, zeros, mode="strict", scheme=IOB2, zero_division=0))
RUN_TRAINING = DEVICE == "cuda"
trainer = None
if RUN_TRAINING:
    set_seed(SEED)
    model = AutoModelForTokenClassification.from_pretrained(MODEL_ID, num_labels=len(tag_names),
        id2label=dict(enumerate(tag_names)), label2id=tag2id)
    args = TrainingArguments(output_dir="checkpoints_ner", num_train_epochs=5, learning_rate=5e-5,
        per_device_train_batch_size=4, per_device_eval_batch_size=4, eval_strategy="epoch", save_strategy="epoch",
        save_total_limit=1, load_best_model_at_end=True, metric_for_best_model="entity_f1", greater_is_better=True,
        fp16=True, bf16=False, report_to="none", seed=SEED, data_seed=SEED, logging_steps=4,
        dataloader_num_workers=0, push_to_hub=False)
    trainer = Trainer(model=model, args=args, train_dataset=ner_ds["train"], eval_dataset=ner_ds["validation"],
                      processing_class=tokenizer, data_collator=DataCollatorForTokenClassification(tokenizer),
                      compute_metrics=ner_metrics)
    trainer.train()
    results["validation"] = trainer.evaluate()
else:
    print("Entraînement NER non exécuté. Continuer les contrôles d'annotation et d'alignement.")


## 4. Lire les erreurs, proposer la prochaine annotation

Après sélection sur validation, exécuter le test une seule fois. Repérer une erreur de type,
une erreur de frontière et une entité inconnue. Si le modèle n’en commet pas dans ce très petit
échantillon, construire des cas nouveaux, sans les intégrer rétroactivement au score.

**Rendu.** Tableau de cinq tokens alignés, explication de -100, F1 entité mesuré, trois besoins
d’annotation. Proposer ce qu’il faudrait changer pour des noms composés, des entités imbriquées
et des références très longues. BIO simple ne représente pas toutes ces structures.


In [ ]:
comparisons = []
if trainer is not None:
    prediction = trainer.predict(ner_ds["test"])
    results["test"] = prediction.metrics
    pred_ids = prediction.predictions.argmax(-1)
    original_test = ner_frame[ner_frame.split == "test"].to_dict(orient="records")
    for original, predicted, gold in zip(original_test, pred_ids, prediction.label_ids):
        valid = [int(p) for p, y in zip(predicted, gold) if y != -100]
        for word, truth, guess in zip(original["tokens"], original["ner_tags"], valid):
            comparisons.append({"mot": word, "attendu": tag_names[truth], "predit": tag_names[guess]})
    display(pd.DataFrame(comparisons))
    print(results["test"])
else:
    results["test"] = None
results["aligned_example"] = {"words": example["tokens"], "labels": aligned}
results["predictions"] = comparisons
conclusion_binome = "À compléter : alignement, métrique pertinente, données à annoter ensuite."


### Repères formateur — éléments de correction

Le score strict pénalise une entité partielle même si un des mots est juste. Le témoin tout-O doit avoir F1=0 dès qu’il y a des entités réelles. Avec quatre phrases de test, préférer une lecture détaillée et un plan d’annotation à une conclusion générale. Des labels BIO plats ne codent pas les entités imbriquées ; expliquer la limite au lieu d’ajouter un I arbitraire.


## Ressources pour prolonger

- [HF — Classification de tokens et alignement](https://huggingface.co/docs/transformers/v4.56.2/en/tasks/token_classification)
- [Seqeval — mode strict et IOB2](https://github.com/chakki-works/seqeval)

Liens et API consultés le 4 octobre 2026. Les exercices et données de ce notebook sont originaux. Les téléchargements de modèles et l’exécution Colab doivent être vérifiés avant la séance.


## Trace à rendre

Compléter vos réponses et ajouter une observation appuyée par une sortie réelle.
Exécuter la dernière cellule, télécharger le JSON dans le panneau Fichiers de Colab,
puis télécharger votre notebook avec ses sorties. Les fichiers du runtime sont temporaires.
Ne pas remplacer une mesure absente par un chiffre plausible. Indiquer les étapes non exécutées.


In [ ]:
results["notebook"] = '04_j3_ner'
results["conclusion_binome"] = globals().get("conclusion_binome", "À compléter")

result_file = OUTPUT / "j3_ner_resultats.json"
result_file.write_text(json.dumps(results, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("Fichier à télécharger :", result_file.resolve())
